# Chapitre 18 - Exercices 18.6 à 18.10

- 18.6 : l'analyse discriminante régularisée calculée en dimension $N$ ;
- 18.7 : moindres carrés quand $p\gg N$ et solution de norme minimale ;
- 18.8 : directions d'« empilement » des données (*data piling*) ;
- 18.9 : empilement maximal contre hyperplan séparateur optimal (SVM) ;
- 18.10 : limite de l'analyse discriminante ridge et empilement maximal.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl import ch18_grande_dimension as g

## Exercice 18.6 : Analyse discriminante régularisée en dimension $N$

**Énoncé (reformulé).** Montrer comment le théorème de la section 18.3.5 (calculs en
dimension $N$ via la SVD) s'applique à l'analyse discriminante régularisée
(section 4.3.1 et éq. (18.9)) :
$\hat\Sigma(\gamma) = \gamma\hat\Sigma + (1-\gamma)\operatorname{diag}(\hat\Sigma)$.

**Rappels.** La LDA (section 4.3) classe $x$ selon le maximum des fonctions
discriminantes
$$\delta_k(x) = x^T\Sigma^{-1}\mu_k - \tfrac12\mu_k^T\Sigma^{-1}\mu_k + \log\pi_k ,$$
où $\mu_k$ est le centroïde de la classe $k$ et $\Sigma$ la covariance intra-classe
commune, estimée par $\hat\Sigma = \frac1{N-K}\sum_i(x_i - \hat\mu_{g_i})(x_i-\hat\mu_{g_i})^T$.
La RDA remplace $\hat\Sigma$ par $\hat\Sigma(\gamma)$, inversible pour $\gamma < 1$.
Inverser directement une matrice $p\times p$ coûte $O(p^3)$.

**Solution.**

*Étape 1 : standardisation.* Soit $\hat D = \operatorname{diag}(\hat\Sigma) = \operatorname{diag}(s_1^2,\dots,s_p^2)$.
Posons $\tilde x = \hat D^{-1/2}x$ (chaque variable divisée par $s_j$),
$\tilde\mu_k = \hat D^{-1/2}\hat\mu_k$ et $\tilde\Sigma = \hat D^{-1/2}\hat\Sigma\hat D^{-1/2}$
(une matrice de corrélation intra-classe). Alors
$\hat\Sigma(\gamma) = \hat D^{1/2}\big(\gamma\tilde\Sigma + (1-\gamma)I\big)\hat D^{1/2}$ et
$$x^T\hat\Sigma(\gamma)^{-1}\hat\mu_k = \tilde x^T\big(\gamma\tilde\Sigma + (1-\gamma)I\big)^{-1}\tilde\mu_k ,$$
et de même pour le terme quadratique : sur les données standardisées, la RDA est une
LDA avec une covariance « ridge » $\gamma\tilde\Sigma + (1-\gamma)I$ - une pénalité
quadratique isotrope.

*Étape 2 : tout vit dans l'espace des lignes.* Soit $\tilde{\mathbf X} = \mathbf R\mathbf V^T$
la SVD réduite des données standardisées ($\mathbf V$ : $p\times N$, $\mathbf V^T\mathbf V = I_N$)
et $r_i = \mathbf V^T\tilde x_i$, de sorte que $\tilde x_i = \mathbf Vr_i$. Alors :

- les centroïdes sont des moyennes de lignes : $\tilde\mu_k = \mathbf V\bar r_k$ ;
- les lignes centrées par classe aussi : $\tilde x_i - \tilde\mu_{g_i} = \mathbf V(r_i - \bar r_{g_i})$ ;
- donc $\tilde\Sigma = \mathbf V\mathbf S\mathbf V^T$ avec
  $\mathbf S = \frac1{N-K}\sum_i(r_i-\bar r_{g_i})(r_i-\bar r_{g_i})^T$ ($N\times N$).

*Étape 3 : inversion en dimension $N$.* Notons $\mathbf A = \gamma\mathbf S + (1-\gamma)I_N$.
Pour tout $a\in\mathbb R^N$,
$$\big(\gamma\mathbf V\mathbf S\mathbf V^T + (1-\gamma)I_p\big)\mathbf V\mathbf A^{-1}a = \mathbf V(\gamma\mathbf S + (1-\gamma)I)\mathbf A^{-1}a = \mathbf Va,$$
(on a utilisé $\mathbf V^T\mathbf V = I_N$), donc
$\big(\gamma\tilde\Sigma + (1-\gamma)I\big)^{-1}\mathbf Va = \mathbf V\mathbf A^{-1}a$.
Pour un point test, $\tilde x_0 = \mathbf Vr_0 + \tilde x_\perp$ avec
$r_0 = \mathbf V^T\tilde x_0$ et $\mathbf V^T\tilde x_\perp = 0$ ; d'où
$$\delta_k(x_0) = r_0^T\mathbf A^{-1}\bar r_k - \tfrac12\bar r_k^T\mathbf A^{-1}\bar r_k + \log\pi_k .$$
C'est une **LDA ordinaire sur les $N$ coordonnées $r_i$**, avec la covariance
régularisée $\mathbf A$. Le coût passe de $O(p^3)$ à $O(N^2p)$. $\square$

> **Remarque.** C'est exactement l'esprit du théorème (18.16)-(18.17) : la
> régularisation est invariante par rotation (après standardisation), et la composante
> des points hors de l'espace des données ne change pas le classement.

In [2]:
def rda_reduced(Xtr, ytr, Xte, gammas):
    """RDA (18.9) calculée en dimension N ; retourne les classes prédites pour chaque gamma."""
    classes, counts, means, s = g.class_statistics(Xtr, ytr)
    N, K = len(ytr), len(classes)
    R, V = g.svd_reduce(Xtr / s)
    inv = np.searchsorted(classes, ytr)
    rbar = np.array([R[inv == k].mean(0) for k in range(K)])
    W = R - rbar[inv]
    S = W.T @ W / (N - K)
    r0 = (Xte / s) @ V
    logpi = np.log(counts / N)
    preds = []
    for gam in gammas:
        A = gam * S + (1 - gam) * np.eye(R.shape[1])
        Ainv_r = np.linalg.solve(A, rbar.T)                    # (N, K)
        delta = r0 @ Ainv_r - 0.5 * np.sum(rbar.T * Ainv_r, 0) + logpi
        preds.append(classes[np.argmax(delta, 1)])
    return preds


def rda_direct(Xtr, ytr, Xte, gamma):
    """RDA par inversion explicite de la matrice p x p (pour vérification)."""
    classes, counts, means, s = g.class_statistics(Xtr, ytr)
    N, K = len(ytr), len(classes)
    W = Xtr - means[np.searchsorted(classes, ytr)]
    Sig = W.T @ W / (N - K)
    Sg = gamma * Sig + (1 - gamma) * np.diag(np.diag(Sig))
    B = np.linalg.solve(Sg, means.T)
    delta = Xte @ B - 0.5 * np.sum(means.T * B, 0) + np.log(counts / N)
    return classes[np.argmax(delta, 1)], delta

**Vérification** sur NCI (8 classes, découpage des cours), d'abord sur 1000 gènes où
l'on peut encore inverser la matrice $1000\times1000$.

In [3]:
X_all, lab_all = datasets.load_nci()
types, counts = np.unique(lab_all, return_counts=True)
keep = np.isin(lab_all, types[counts >= 5])
X, lab = X_all[keep], lab_all[keep]
split = g.stratified_folds(lab, 3, np.random.RandomState(0))
te, tr = split == 0, split != 0
Xtr, ytr, Xte, yte = X[tr], lab[tr], X[te], lab[te]

sub = slice(0, 1000)
for gam in [0.0, 0.3, 0.9]:
    p_dir, _ = rda_direct(Xtr[:, sub], ytr, Xte[:, sub], gam)
    p_red = rda_reduced(Xtr[:, sub], ytr, Xte[:, sub], [gam])[0]
    print(f"gamma = {gam} : prédictions identiques ? {np.all(p_dir == p_red)}")

gamma = 0.0 : prédictions identiques ? True
gamma = 0.3 : prédictions identiques ? True
gamma = 0.9 : prédictions identiques ? True


On peut aussi comparer les **scores** eux-mêmes, au terme $-\frac12\tilde x_0^T(\dots)^{-1}\tilde x_0$
près (commun à toutes les classes). Passons maintenant aux 6830 gènes, uniquement en
dimension réduite.

In [ ]:
gammas = np.r_[0.0, np.linspace(0.05, 0.95, 19), 0.99]
preds = rda_reduced(Xtr, ytr, Xte, gammas)
err = [np.mean(p != yte) for p in preds]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(gammas, err, "o-", color=BLUE)
ax.set_xlabel("γ (0 = LDA diagonale, → 1 = LDA complète)")
ax.set_ylabel("taux d'erreur de test")
ax.set_title("RDA sur NCI, 6830 gènes, calculée en dimension 39")
plt.show()
print("LDA diagonale (s0 = 0) via les centroïdes réduits :",
      np.mean(g.NearestShrunkenCentroids(0.0, s0=0.0).fit(Xtr, ytr).predict(Xte) != yte).round(3),
      " ; RDA gamma = 0 :", round(err[0], 3))

Pour $\gamma = 0$, on retrouve la LDA diagonale (même erreur que les centroïdes réduits
sans rétrécissement ni $s_0$). Dès que $\gamma > 0$, l'erreur tombe à 4 erreurs sur 18
et reste constante sur toute la plage : comme dans le tableau 18.1 du livre (où toutes
les valeurs $\gamma\in(0.002, 0.55)$ donnaient la même erreur), le résultat est peu
sensible au choix de $\gamma$.

> **À retenir.** Après standardisation, la RDA est une LDA à covariance « ridge » ;
> son calcul se ramène à une LDA sur les $N$ coordonnées de la SVD.

## Exercice 18.7 : Moindres carrés quand $p\gg N$

**Énoncé (reformulé).** On suppose $p\gg N$ et $\operatorname{rang}(\mathbf X) = N$. Soit
$\mathbf X = \mathbf U\mathbf D\mathbf V^T = \mathbf R\mathbf V^T$ la SVD, avec
$\mathbf R$ ($N\times N$) inversible et $\mathbf V$ ($p\times N$) à colonnes orthonormées.

(a) Montrer qu'il existe une infinité de solutions des moindres carrés, toutes à
résidus nuls.
(b) Montrer que l'estimateur ridge s'écrit $\hat\beta_\lambda = \mathbf V(\mathbf R^T\mathbf R+\lambda I)^{-1}\mathbf R^Ty$ (éq. (18.56)).
(c) Montrer que pour $\lambda = 0$, la solution $\hat\beta_0 = \mathbf V\mathbf D^{-1}\mathbf U^Ty$
a des résidus nuls et qu'elle est l'unique solution de norme euclidienne minimale
parmi les solutions à résidus nuls.